# 🔄 State Management com StateMachine

Agentes de IA não são chamadas únicas — são **workflows** que executam múltiplos passos, tomam decisões e mantêm contexto entre etapas. Este notebook mostra como a `StateMachine` formaliza essa estrutura.

---

## 🗺️ Mapa do Notebook

| Seção | Conceito | Padrão Aprendido |
|---|---|---|
| **1. Conceitos Básicos** | Schema, Steps, Conexões, Snapshots | Fluxo linear sequencial |
| **2. Routing e Loops** | Router functions, grafos cíclicos | Iteração condicional |

---

## 🧠 Conceito Central

$$\text{StateMachine} = \text{Grafo Dirigido} + \text{Estado Compartilhado} + \text{Navegação por Router}$$

Uma máquina de estados é um **grafo** onde:
- Os **nós** são `Steps` (funções que transformam o estado)
- As **arestas** são `Transitions` (quem executa após quem)
- O **estado** é um dicionário tipado (Schema) que flui e evolui por todos os nós

> **Conexão com agentes reais:** Frameworks como LangGraph e AutoGen são, em essência, máquinas de estado sofisticadas. Entender a `StateMachine` deste notebook é entender a fundação de qualquer orquestrador de agentes.

---

## Componentes da StateMachine

```mermaid
flowchart LR
    EP["🚪 EntryPoint\n__entry__"]
    S1["📦 Step\n(nome + função)"]
    S2["📦 Step\n(nome + função)"]
    T["🏁 Termination\n__termination__"]

    EP -->|"Transition"| S1
    S1 -->|"Transition"| S2
    S2 -->|"Transition"| T
```

| Componente | Papel |
|---|---|
| `EntryPoint` | Nó especial de entrada — injeta o estado inicial |
| `Step` | Nó de processamento — recebe estado, retorna estado modificado |
| `Termination` | Nó especial de saída — encerra a execução |
| `Transition` | Aresta do grafo — define para qual nó ir após cada step |

## O que vamos aprender:
- Conceitos e implementação de máquinas de estado
- Criação e conexão de steps de workflow
- Gerenciamento de transições e fluxo de dados
- Trabalho com routing e loops em máquinas de estado
- Entendimento de snapshots e fluxo de execução

### Setup

In [1]:
from typing import TypedDict
from lib.state_machine import (
    StateMachine,
    Step,
    EntryPoint,
    Termination,
)

## 1. Conceitos Básicos da StateMachine

Vamos construir um workflow simples em **4 etapas**:

```mermaid
flowchart LR
    subgraph "Construção do Workflow"
        direction TB
        A["① Definir Schema\n(TypedDict)"]
        B["② Criar StateMachine\n(Schema)"]
        C["③ Criar Steps\n(funções + nomes)"]
        D["④ Conectar\n(workflow.connect)"]
    end
    A --> B --> C --> D
```

### O Schema: Contrato do Estado

O `Schema` é um `TypedDict` que define **todas as chaves** que podem existir no estado. Ele serve como:
- **Documentação:** Deixa claro quais dados o workflow carrega
- **Filtragem:** A `StateMachine` descarta chaves que não pertencem ao schema (sem quebrar)
- **Tipagem estática:** IDEs e linters podem verificar acesso incorreto a campos

> **Analogia:** O Schema é como a tabela de um banco de dados — define as colunas permitidas. Steps só podem ler e escrever colunas que existam nessa tabela.

**Creating the Schema and the State Machine**

In [ ]:
class Schema(TypedDict):
    """Schema defining the structure of our state.
    
    Attributes:
        input: The input value to process
        output: The processed output value
    """
    input: int
    output: int

In [ ]:
# Create our state machine instance
workflow = StateMachine(Schema)

**Defining the logic for Steps**

In [ ]:
def step_input(state: Schema) -> Schema:
    """First step: Increment the input value.
    
    Args:
        state: Current state containing input value
        
    Returns:
        Updated state with incremented value in output
    """
    return {"output": state["input"] + 1, "random": 10}



In [ ]:
def step_double(state: Schema) -> Schema:
    """Second step: Double the previous output.
    
    Args:
        state: Current state containing output from previous step
        
    Returns:
        Updated state with doubled output value
    """
    return {"output": state["output"] * 2}


**Creating and Connecting Steps**

In [5]:
entry = EntryPoint()
s1 = Step("input", step_input)
s2 = Step("double", step_double)
termination = Termination()

In [6]:
workflow.add_steps([entry, s1, s2, termination])

In [7]:
workflow.connect(entry, s1)
workflow.connect(s1, s2)
workflow.connect(s2, termination)

**Inspecionando as Transições**

Cada chamada a `workflow.connect(origem, destino)` registra uma **aresta** no grafo interno da máquina de estados. O dicionário `workflow.transitions` mapeia o nome de cada nó de origem para a lista de transições possíveis.

O grafo linear acima pode ser visualizado como:

```
__entry__ ──► input ──► double ──► __termination__
```

> **Por que isso importa?** A `StateMachine` não executa os passos em sequência cega — ela *navega* o grafo. Isso permite ramificações e loops, como veremos na seção avançada.

In [8]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['input'])],
 'input': [Transition('input' -> ['double'])],
 'double': [Transition('double' -> ['__termination__'])]}

**Running the Workflow**

In [9]:
initial_state = {"input": 4}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: input
[StateMachine] Executing step: double
[StateMachine] Terminating: __termination__


Run('0e3c6b52-8121-4378-a44e-49785aaa5888')

In [10]:
run_object.snapshots

[__entry__.Snapshot({'input': 4}),
 input.Snapshot({'input': 4, 'output': 5}),
 double.Snapshot({'input': 4, 'output': 10})]

**Entendendo os Snapshots**

Cada `Snapshot` representa uma **fotografia imutável** do estado após a execução de um passo. Observe os valores acumulados:

| Passo | `input` | `output` | O que aconteceu |
|---|---|---|---|
| `__entry__` | 4 | — | Estado inicial injetado |
| `input` | 4 | 5 | `output = input + 1` → `4 + 1 = 5` |
| `double` | 4 | 10 | `output = output * 2` → `5 × 2 = 10` |

> **Padrão de merge:** Os steps retornam apenas as **chaves que modificaram**. A `StateMachine` faz o merge com o estado anterior — por isso `input` continua visível em todos os snapshots, mesmo que `step_double` não o toque.

> **Chaves extras são ignoradas:** `step_input` retornou `{"output": 5, "random": 10}`. A chave `random` não pertence ao `Schema` e é **descartada silenciosamente** — a máquina valida contra o schema definido.

## 2. Routing e Loops — Controle de Fluxo Avançado

Fluxos lineares são limitados. Agentes reais precisam de **ramificações** (tomar decisões) e **loops** (repetir até uma condição ser satisfeita).

### Padrões de Fluxo em Agentes

```mermaid
flowchart TD
    subgraph "Fluxo Linear (Seção 1)"
        direction LR
        E1[entry] --> A[input] --> B[double] --> T1[termination]
    end

    subgraph "Loop com Router (Seção 2)"
        direction LR
        E2[entry] --> INC[increment]
        INC -->|"count < max"| INC
        INC -->|"count >= max"| T2[termination]
    end
```

| Padrão | Quando usar | Exemplo em agentes |
|---|---|---|
| **Linear** | Passos independentes e sequenciais | Extract → Transform → Load |
| **Loop + Router** | Refinamento iterativo até critério | Retry até LLM retornar JSON válido |
| **Branching** | Decisões baseadas no estado | Roteamento por intenção do usuário |

### Como o Router Funciona

```python
# A assinatura do router é sempre: (state: Schema) -> Step
def check_counter(state: CounterSchema) -> Step:
    if state["count"] >= state["max_value"]:
        return termination   # Navega para __termination__
    return increment          # Navega de volta para si mesmo
```

O router é uma **função de decisão pura** — recebe o estado atual e retorna o **próximo nó** a executar. Isso desacopla a lógica de controle da lógica de negócio dos steps.

> **Conexão com LangGraph:** O conceito de "conditional edge" do LangGraph é exatamente este padrão — uma função que recebe o estado e retorna o nome do próximo nó.

In [ ]:
class CounterSchema(TypedDict):
    """Schema for a counter-based workflow.
    
    Attributes:
        count: Current counter value
        max_value: Maximum value before termination
    """
    count: int
    max_value: int

In [12]:
workflow = StateMachine(CounterSchema)

In [ ]:
def increment_counter(state: CounterSchema) -> CounterSchema:
    """Increment the counter value.
    
    Args:
        state: Current state with counter value
        
    Returns:
        Updated state with incremented counter
    """
    return {"count": state["count"] + 1}

In [14]:
# Create steps
entry = EntryPoint()
increment = Step("increment", increment_counter)
termination = Termination()

In [15]:
workflow.add_steps([entry, increment, termination])

In [ ]:
# Router logic
def check_counter(state: CounterSchema) -> Step:
    """Determine next step based on counter value.
    
    Args:
        state: Current state with counter and max value
        
    Returns:
        Next step to execute (increment or terminate)
    """
    if state["count"] >= state["max_value"]:
        return termination
    return increment

In [17]:
# Connect steps with a loop in increment
workflow.connect(entry, increment)
workflow.connect(increment, [increment, termination], check_counter)

**Conectando com um Router (Decisão Condicional)**

Quando `workflow.connect(origem, [destino_A, destino_B], router_fn)` é chamado com uma **lista de destinos** e uma **função de roteamento**, a máquina de estados delega a decisão de qual próximo passo executar à função `router_fn`.

```
increment ──► check_counter ──► increment  (se count < max_value)
                           └──► __termination__  (se count >= max_value)
```

| Assinatura do Router | Responsabilidade |
|---|---|
| `check_counter(state) -> Step` | Recebe o estado atual e retorna o próximo `Step` a executar |

> **Loop controlado:** A `StateMachine` suporta ciclos no grafo — um passo pode apontar para si mesmo. A condição de parada é inteiramente responsabilidade da função de roteamento.

In [18]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['increment'])],
 'increment': [Transition('increment' -> ['increment', '__termination__'])]}

In [19]:
initial_state = {"count": 0, "max_value": 3}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Terminating: __termination__


Run('33630c81-9b65-42ce-b406-60523de03622')

In [20]:
run_object.snapshots

[__entry__.Snapshot({'count': 0, 'max_value': 3}),
 increment.Snapshot({'count': 1, 'max_value': 3}),
 increment.Snapshot({'count': 2, 'max_value': 3}),
 increment.Snapshot({'count': 3, 'max_value': 3})]

In [22]:
run_object.get_final_state()

{'count': 3, 'max_value': 3}

**Resumo: Fluxo do Contador**

`get_final_state()` retorna o **último snapshot** acumulado — equivalente a ler `run_object.snapshots[-1]`.

Rastreando a execução com `max_value = 3`:

| Iteração | `count` antes | Decisão do Router | `count` depois |
|---|---|---|---|
| 1ª | 0 | `0 < 3` → continua | 1 |
| 2ª | 1 | `1 < 3` → continua | 2 |
| 3ª | 2 | `2 < 3` → continua | 3 |
| — | 3 | `3 >= 3` → termina | — |

> **Comparação com a seção básica:** No fluxo linear (`input → double`), os **dados se transformam** a cada passo. No loop com router, os **dados se acumulam iterativamente** até uma condição ser satisfeita — padrão comum em agentes que precisam de múltiplas tentativas ou refinamentos.